# Bi-GRU + Max/Attention Pooling — Train, đánh giá, lưu kết quả

Notebook này train model **Bi-GRU** (`toxic_comments.models.bigru`) trên dữ liệu Jigsaw đã clean, in kết quả và lưu toàn bộ đầu ra vào một thư mục riêng cho mỗi lần chạy:

```
models/bigru/<RUN_NAME>/            # model .joblib (mỗi fold một file)
results/bigru/<RUN_NAME>/           # config, metrics, per-label, loss, biểu đồ, dự đoán mẫu
```

**Yêu cầu:** đã chạy `add_bigru.py` (hoặc `add_bigru.sh`), và có file `data/processed/train_clean.csv` (tạo bằng `01_cleaning.ipynb`).

Chỉ cần sửa ô **2. Cấu hình**, sau đó chạy `Run All`.

## 0. Kiểm tra GPU

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"✅ Dùng GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  KHÔNG CÓ GPU — Bi-GRU trên full data sẽ rất chậm. Nên chạy trên RunPod/Colab, "
          "hoặc đặt SAMPLE_SIZE ở ô cấu hình để chạy thử.")

## 1. Import và đường dẫn

In [ ]:
import json
import sys
import time
from datetime import datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from toxic_comments.config import (
    EMBEDDINGS_DIR,
    HEAVY_TEXT_COLUMN,
    K_FOLD_DATA_DIR,
    LABEL_COLUMNS,
    MODELS_DIR,
    PROCESSED_DATA_DIR,
    RESULTS_DIR,
)
from toxic_comments.cleaning import clean_heavy, clean_light
from toxic_comments.embeddings.fasttext import FASTTEXT_VECTOR_FILE
from toxic_comments.embeddings.glove import GLOVE_TWITTER_VECTOR_FILE
from toxic_comments.splits import save_kfold_datasets, train_validation_test_split
from toxic_comments.thresholds import ThresholdedClassifier, thresholds_to_frame
from toxic_comments.train import evaluate_model_detailed, save_model

try:
    from toxic_comments.models.bigru import build_bigru_max_attention_classifier
    build_bigru_max_attention_classifier  # noqa: B018
except ImportError as exc:
    raise ImportError("Chưa có Bi-GRU trong repo — hãy chạy `python add_bigru.py` ở thư mục gốc trước.") from exc

print("PROJECT_ROOT =", PROJECT_ROOT)

## 2. Cấu hình

| Tham số | Ý nghĩa |
|---|---|
| `RUN_MODE` | `"holdout"`: chia train/val/test 80/10/10 (nhanh). `"folds"`: 5-fold CV, dùng chung `data/k_fold/` với các model khác để so sánh công bằng. |
| `EMBEDDING` | `"fasttext"`, `"glove"` hoặc `"random"` (học embedding từ đầu, không cần tải file). |
| `SAMPLE_SIZE` | `None` = full data. Đặt số (ví dụ `5000`) để chạy thử nhanh; khi đó folds được tạo riêng trong thư mục run, **không** ghi đè `data/k_fold/`. |
| `TUNE_THRESHOLDS` | Tìm ngưỡng tối ưu F1 cho từng nhãn (train 2 lần mỗi fold: 1 lần để tune, 1 lần trên toàn bộ). |
| `HPARAMS` | Siêu tham số của Bi-GRU. |

In [ ]:
RUN_MODE = "holdout"          # "holdout" | "folds"
EMBEDDING = "fasttext"        # "fasttext" | "glove" | "random"
SAMPLE_SIZE = None            # None = full data; ví dụ 5000 để chạy thử
TUNE_THRESHOLDS = True
N_FOLDS = 5
RANDOM_STATE = 42

HPARAMS = dict(
    max_length=200,           # số token tối đa mỗi comment
    hidden_size=64,           # mỗi chiều GRU -> output 2*64
    num_layers=2,             # số lớp Bi-GRU xếp chồng
    spatial_dropout=0.2,
    dropout=0.1,
    batch_size=256,
    learning_rate=1e-3,
    num_epochs=4,
    freeze_embeddings=True,   # chỉ áp dụng khi dùng embedding pretrained
    use_pos_weight=False,     # True: tăng trọng số nhãn hiếm (threat, identity_hate) trong BCE
    max_embedding_vectors=None,
    verbose=True,
)

RUN_NAME = f"bigru_{EMBEDDING}_{RUN_MODE}_{datetime.now():%Y%m%d_%H%M%S}"


In [ ]:
EMBEDDING_PATHS = {
    "fasttext": EMBEDDINGS_DIR / FASTTEXT_VECTOR_FILE,
    "glove": EMBEDDINGS_DIR / GLOVE_TWITTER_VECTOR_FILE,
    "random": None,
}
if EMBEDDING not in EMBEDDING_PATHS:
    raise ValueError(f"EMBEDDING phải là một trong {list(EMBEDDING_PATHS)}")
EMBEDDING_PATH = EMBEDDING_PATHS[EMBEDDING]
if EMBEDDING_PATH is not None and not EMBEDDING_PATH.exists():
    raise FileNotFoundError(
        f"Không thấy file embedding: {EMBEDDING_PATH}\n"
        "Tải file và đặt đúng chỗ, hoặc đặt EMBEDDING = 'random' để chạy không cần embedding."
    )

RUN_MODEL_DIR = MODELS_DIR / "bigru" / RUN_NAME
RUN_RESULT_DIR = RESULTS_DIR / "bigru" / RUN_NAME
RUN_MODEL_DIR.mkdir(parents=True, exist_ok=True)
RUN_RESULT_DIR.mkdir(parents=True, exist_ok=True)

config = dict(
    run_name=RUN_NAME, run_mode=RUN_MODE, embedding=EMBEDDING,
    embedding_path=str(EMBEDDING_PATH) if EMBEDDING_PATH else None,
    sample_size=SAMPLE_SIZE, tune_thresholds=TUNE_THRESHOLDS, n_folds=N_FOLDS,
    random_state=RANDOM_STATE, device=DEVICE, text_column=HEAVY_TEXT_COLUMN,
    hparams=HPARAMS, started_at=datetime.now().isoformat(timespec="seconds"),
)
(RUN_RESULT_DIR / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
print("Model sẽ lưu tại :", RUN_MODEL_DIR)
print("Kết quả lưu tại  :", RUN_RESULT_DIR)

## 3. Load data đã clean

In [ ]:
data_path = PROCESSED_DATA_DIR / "train_clean.csv"
if not data_path.exists():
    raise FileNotFoundError(f"Không thấy {data_path}. Chạy notebooks/01_cleaning.ipynb trước.")

data = pd.read_csv(data_path)
data[HEAVY_TEXT_COLUMN] = data[HEAVY_TEXT_COLUMN].fillna("").astype(str)
if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(data):
    data = data.sample(SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"Số dòng: {len(data):,}")
print("Tỉ lệ positive từng nhãn:")
print((data[LABEL_COLUMNS].mean() * 100).round(2).astype(str) + " %")
data[[HEAVY_TEXT_COLUMN, *LABEL_COLUMNS]].head(3)

## 4. Chia dữ liệu

In [ ]:
def load_or_create_folds() -> list[tuple[str, pd.DataFrame, pd.DataFrame]]:
    # Full data: dùng chung data/k_fold/ với cả nhóm. Sample: tạo folds riêng trong thư mục run.
    folds_dir = K_FOLD_DATA_DIR if SAMPLE_SIZE is None else RUN_RESULT_DIR / "folds"
    existing = sorted(p for p in folds_dir.glob("fold_*") if (p / "train.csv").exists() and (p / "test.csv").exists())
    if not existing:
        save_kfold_datasets(data=data, output_dir=folds_dir, n_splits=N_FOLDS, random_state=RANDOM_STATE,
                            text_column=HEAVY_TEXT_COLUMN, strategy="stratified")
        print(f"Đã tạo {N_FOLDS} folds tại {folds_dir}")
    else:
        print(f"Dùng {len(existing)} folds có sẵn tại {folds_dir}")
    folds = []
    for fold_dir in sorted(folds_dir.glob("fold_*")):
        if (fold_dir / "train.csv").exists() and (fold_dir / "test.csv").exists():
            tr = pd.read_csv(fold_dir / "train.csv")
            te = pd.read_csv(fold_dir / "test.csv")
            for df in (tr, te):
                df[HEAVY_TEXT_COLUMN] = df[HEAVY_TEXT_COLUMN].fillna("").astype(str)
            folds.append((fold_dir.name, tr, {"test": te}))
    total_test = sum(len(s["test"]) for _, _, s in folds)
    if total_test != len(data):
        print(f"⚠️  Tổng số dòng test của các fold ({total_test:,}) khác số dòng data ({len(data):,}) — "
              "folds có thể được tạo từ phiên bản train_clean.csv khác.")
    return folds


if RUN_MODE == "holdout":
    train_df, val_df, test_df = train_validation_test_split(
        data, validation_size=0.1, test_size=0.1, random_state=RANDOM_STATE, stratified=True
    )
    SPLITS = [("holdout", train_df, {"validation": val_df, "test": test_df})]
elif RUN_MODE == "folds":
    SPLITS = load_or_create_folds()
else:
    raise ValueError("RUN_MODE phải là 'holdout' hoặc 'folds'")

for name, tr, evals in SPLITS:
    print(f"{name}: train={len(tr):,} | " + " | ".join(f"{k}={len(v):,}" for k, v in evals.items()))

## 5. Train

Mỗi split: build model → fit → đánh giá → lưu model. Metric được ghi ra CSV **ngay sau mỗi fold**, nên nếu bị ngắt giữa chừng thì các fold đã xong vẫn còn.

In [ ]:
def build_estimator():
    model = build_bigru_max_attention_classifier(
        embedding_path=EMBEDDING_PATH, device=DEVICE, random_state=RANDOM_STATE, **HPARAMS
    )
    return ThresholdedClassifier(model, random_state=RANDOM_STATE) if TUNE_THRESHOLDS else model


def unwrap(estimator):
    # Trả về Bi-GRU đã fit bên trong (ThresholdedClassifier giữ nó ở .estimator_).
    return getattr(estimator, "estimator_", estimator)


metrics_path = RUN_RESULT_DIR / "metrics.csv"
per_label_path = RUN_RESULT_DIR / "per_label.csv"
for p in (metrics_path, per_label_path):
    p.unlink(missing_ok=True)

metric_rows, per_label_frames, loss_rows, threshold_frames, model_paths = [], [], [], [], []

for split_name, train_part, eval_sets in SPLITS:
    print(f"\n{'=' * 70}\n▶ {split_name}: train {len(train_part):,} dòng\n{'=' * 70}")
    estimator = build_estimator()
    start = time.perf_counter()
    estimator.fit(train_part[HEAVY_TEXT_COLUMN], train_part[LABEL_COLUMNS].to_numpy())
    train_seconds = time.perf_counter() - start

    for eval_name, eval_df in eval_sets.items():
        aggregate, per_label = evaluate_model_detailed(estimator, eval_df, text_column=HEAVY_TEXT_COLUMN)
        row = {"run_name": RUN_NAME, "model_name": f"bigru_{EMBEDDING}", "fold": split_name,
               "split": eval_name, "rows": len(eval_df), "train_seconds": round(train_seconds, 1), **aggregate}
        metric_rows.append(row)
        per_label.insert(0, "split", eval_name)
        per_label.insert(0, "fold", split_name)
        per_label_frames.append(per_label)
        pd.DataFrame([row]).to_csv(metrics_path, mode="a", header=not metrics_path.exists(), index=False)
        per_label.to_csv(per_label_path, mode="a", header=not per_label_path.exists(), index=False)
        print(f"  [{eval_name}] macro_F1={aggregate['macro_f1']:.4f} | micro_F1={aggregate['micro_f1']:.4f} | "
              f"macro_AUC={aggregate['macro_roc_auc'] if aggregate['macro_roc_auc'] is not None else float('nan'):.4f}")

    inner = unwrap(estimator)
    for epoch, loss in enumerate(inner.loss_history_, start=1):
        loss_rows.append({"fold": split_name, "epoch": epoch, "train_loss": loss})
    if TUNE_THRESHOLDS and getattr(estimator, "thresholds_", None) is not None:
        th = thresholds_to_frame(estimator.thresholds_)
        th.insert(0, "fold", split_name)
        threshold_frames.append(th)

    path = save_model(estimator, RUN_MODEL_DIR / split_name / "bigru.joblib")
    model_paths.append(path)
    print(f"  ⏱  {train_seconds / 60:.1f} phút | 💾 {path}")

loss_df = pd.DataFrame(loss_rows)
loss_df.to_csv(RUN_RESULT_DIR / "loss_history.csv", index=False)
if threshold_frames:
    pd.concat(threshold_frames, ignore_index=True).to_csv(RUN_RESULT_DIR / "thresholds.csv", index=False)
print("\n✅ Train xong.")

## 6. Kết quả tổng hợp

In [ ]:
metrics_df = pd.DataFrame(metric_rows)
KEY_METRICS = ["macro_f1", "micro_f1", "macro_roc_auc", "micro_roc_auc", "hamming_loss", "subset_accuracy"]
key_cols = [c for c in KEY_METRICS if c in metrics_df.columns]

print("### Metric từng fold / split")
display(metrics_df[["fold", "split", "rows", "train_seconds", *key_cols]].round(4))

summary = (
    metrics_df.groupby("split")[key_cols].agg(["mean", "std"]).round(4)
)
summary.to_csv(RUN_RESULT_DIR / "summary.csv")
print("### Trung bình ± độ lệch chuẩn")
summary

## 7. Kết quả từng nhãn

In [ ]:
per_label_df = pd.concat(per_label_frames, ignore_index=True)
report_split = "test"
per_label_summary = (
    per_label_df[per_label_df["split"] == report_split]
    .groupby("label")[[c for c in ["precision", "recall", "f1", "roc_auc", "support"] if c in per_label_df.columns]]
    .mean()
    .reindex(LABEL_COLUMNS)
    .round(4)
)
per_label_summary.to_csv(RUN_RESULT_DIR / "per_label_summary.csv")
print(f"### Per-label trên tập {report_split} (trung bình qua các fold)")
per_label_summary

## 8. Biểu đồ

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for fold, grp in loss_df.groupby("fold"):
    axes[0].plot(grp["epoch"], grp["train_loss"], marker="o", label=fold)
axes[0].set_title("Train loss theo epoch (model fit cuối cùng)")
axes[0].set_xlabel("Epoch")
axes[0].set_xticks(sorted(loss_df["epoch"].unique()))
axes[0].set_ylabel("BCE loss")
axes[0].legend(fontsize=8)

f1_vals = per_label_summary["f1"]
bars = axes[1].bar(f1_vals.index, f1_vals.values, color="#2E75B6")
axes[1].bar_label(bars, fmt="%.3f", fontsize=8)
axes[1].set_title(f"F1 từng nhãn ({report_split})")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=25)

fig.tight_layout()
fig.savefig(RUN_RESULT_DIR / "plots.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. So sánh với các model khác (nếu đã có kết quả)

Đọc các file `results/*_fold_metrics.csv` / `*_holdout_metrics.csv` mà `toxic_comments.train` đã tạo cho các model khác của nhóm.

In [ ]:
pattern = "*_fold_metrics.csv" if RUN_MODE == "folds" else "*_holdout_metrics.csv"
others = []
for f in sorted(RESULTS_DIR.glob(pattern)):
    df = pd.read_csv(f)
    others.append(df[df["split"] == "test"] if "split" in df.columns else df)

ours = metrics_df[metrics_df["split"] == "test"].assign(model_name=f"bigru_{EMBEDDING} (run này)")
if others:
    combined = pd.concat([*others, ours], ignore_index=True)
    cols = [c for c in ["macro_f1", "micro_f1", "macro_roc_auc", "micro_roc_auc"] if c in combined.columns]
    comparison = combined.groupby("model_name")[cols].mean().round(4).sort_values("macro_f1", ascending=False)
    comparison.to_csv(RUN_RESULT_DIR / "comparison.csv")
    display(comparison)
else:
    print(f"Chưa có file {pattern} của model khác trong {RESULTS_DIR} — bỏ qua so sánh.")

## 10. Load lại model đã lưu và dự đoán comment mới

In [ ]:
model_path = model_paths[0]
loaded = joblib.load(model_path)
print("Đã load:", model_path)

new_comments = [
    "You are such an idiot, I hope something bad happens to you.",
    "Thanks so much for fixing this, really appreciate it!",
    "I will find you and kill you.",
    "This article needs better sources, please add citations.",
]
cleaned = pd.Series(new_comments).apply(clean_light).apply(clean_heavy)
probs = loaded.predict_proba(cleaned)
preds = loaded.predict(cleaned)   # dùng ngưỡng đã tune nếu TUNE_THRESHOLDS=True

pred_df = pd.DataFrame({"comment_text": new_comments})
for i, label in enumerate(LABEL_COLUMNS):
    pred_df[f"{label}_prob"] = probs[:, i].round(4)
    pred_df[f"{label}_pred"] = preds[:, i]
pred_df.to_csv(RUN_RESULT_DIR / "sample_predictions.csv", index=False)
pd.set_option("display.max_colwidth", 45)
pred_df

## 11. Attention: model chú ý vào từ nào?

Trọng số attention trên văn bản **đã clean** (đầu vào thật của model). Đây là gợi ý trực quan, không phải giải thích chắc chắn (Jain & Wallace, 2019).

In [ ]:
bigru = unwrap(loaded)
fig, axes = plt.subplots(len(new_comments), 1, figsize=(10, 1.3 * len(new_comments)))
attention_rows = []
for ax, raw, text in zip(np.atleast_1d(axes), new_comments, cleaned):
    att = bigru.explain(text)
    attention_rows.extend({"comment_text": raw, **r} for r in att.to_dict("records"))
    ax.imshow(att["attention"].to_numpy()[None, :], cmap="Reds", aspect="auto")
    ax.set_xticks(range(len(att)), att["token"], rotation=30, fontsize=8)
    ax.set_yticks([])
    ax.set_title(raw[:80], fontsize=9, loc="left")
fig.tight_layout()
fig.savefig(RUN_RESULT_DIR / "attention_examples.png", dpi=150, bbox_inches="tight")
plt.show()
pd.DataFrame(attention_rows).to_csv(RUN_RESULT_DIR / "attention_examples.csv", index=False)

## 12. Danh sách file đã lưu

In [ ]:
config["finished_at"] = datetime.now().isoformat(timespec="seconds")
(RUN_RESULT_DIR / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")

for folder in (RUN_MODEL_DIR, RUN_RESULT_DIR):
    print(f"📁 {folder.relative_to(PROJECT_ROOT)}")
    for f in sorted(folder.rglob("*")):
        if f.is_file():
            print(f"   {str(f.relative_to(folder)):45s} {f.stat().st_size / 1024:10.1f} KB")